Question:1

In [33]:
import numpy as np

sizes = [4,3,5]
edges= [np.array([[0,1,0,2,0,3],  [1,0,2,0,3,0]]),
        np.array([[0,1,1,2],[1,0,2,1]]),
        np.array([[0,1,1,2,2,3,3,4], [1,0,2,1,3,2,4,3]])]
 # TODO: running offsetss, the concatenated edge list, the batch vector, the segment sums, and a check that no edge joins two molecules
offsets= np.cumsum([0] +sizes[:-1])

src= np.concatenate([e[0] + o for e,o in zip(edges, offsets)])
dst= np.concatenate([e[1] + o for e,o in zip(edges, offsets)])

batch =np.concatenate([np.full(s,i) for i,s in enumerate(sizes)])

N=sum(sizes)
h=np.arange(N)

seg=np.zeros(len(sizes))
np.add.at(seg,batch,h)
cross=np.sum(batch[src]!= batch[dst])
print("Number of atoms:",N)
print("the directed edges:",src.size)
print("batch vec:",batch)
print("offset src:",src)
print("offset dst:",dst)
print("segment sums:",seg)
print("edges joining diff mol:",cross)

Number of atoms: 12
the directed edges: 18
batch vec: [0 0 0 0 1 1 1 2 2 2 2 2]
offset src: [ 0  1  0  2  0  3  4  5  5  6  7  8  8  9  9 10 10 11]
offset dst: [ 1  0  2  0  3  0  5  4  6  5  8  7  9  8 10  9 11 10]
segment sums: [ 6. 15. 45.]
edges joining diff mol: 0


If offsets were omitted, local indices would collide and edges would wrongly connect atoms belonging to different molecules

Quetion:2

In [34]:
import numpy as np

a,b= -1200.0, -300.0 #KJ/mol
ns= np.arange(1,13)
E=a*ns+b

c=E.mean()
err= c-E
slope=np.polyfit(ns,err,1)[0]

print("best constant predictor:", c)
print("signed error at n=1,n=6,n=12:", err[0], err[5], err[11])
print("slope of the error vs n:", slope)

for n_,e_ in zip(ns,E/ns):
  print(f" n={n_:2d} E/n= {e_:.4f}")

best constant predictor: -8100.0
signed error at n=1,n=6,n=12: -6600.0 -600.0 6600.0
slope of the error vs n: 1200.0
 n= 1 E/n= -1500.0000
 n= 2 E/n= -1350.0000
 n= 3 E/n= -1300.0000
 n= 4 E/n= -1275.0000
 n= 5 E/n= -1260.0000
 n= 6 E/n= -1250.0000
 n= 7 E/n= -1242.8571
 n= 8 E/n= -1237.5000
 n= 9 E/n= -1233.3333
 n=10 E/n= -1230.0000
 n=11 E/n= -1227.2727
 n=12 E/n= -1225.0000


A sum readout removes the defect as it scales with molecule size matching the extensive target

Question 3:

In [35]:
import numpy as np
De, alpha,re =4.75, 1.94, 0.741

def E(r):
  return De * (1 - np.exp(-alpha * (r-re))) ** 2

def dE(r):
  ex = np.exp(-alpha*(r-re))
  return 2.0* De * alpha*ex*(1.0- ex)

r=0.9
h=1e-5

central =(E(r+h)-E(r-h))/(2*h)
print ("E(r):",E(r))
print ("analytical diff:",dE(r))
print ("central diff:",central)
print ("diff between analytical and central:", dE(r)-central)

r1=np.array([0.0,0.0,0.0])
r2=np.array([0.9,0.0,0.0])
d2=r2-r1
d1=-d2
rlen= np.linalg.norm(d)
u1=d1/rlen
u2=d2/rlen
F2= -dE(rlen)* u2
F1=-dE(rlen)* u1
print("F1:",F1)
print("F2:",F2)
print("total force:",F1+F2)

E(r): 0.33463365844867393
analytical diff: 3.593361126607759
central diff: 3.5933611249455706
diff between analytical and central: 1.662188608975157e-09
F1: [3.59336113 0.         0.        ]
F2: [-3.59336113 -0.         -0.        ]
total force: [0. 0. 0.]


The vanishing sum expresses Newton's Third Law

Question:4

In [36]:
import numpy as np

scaffold= np.array([0,0,0,0,1,1,1,2,2,2,3,3])
n=len(scaffold)

idx= np.random.default_rng(3).permutation(n)

test_r=np.sort(idx[:4])
train_r=np.sort(idx[4:])

def count(test,train):
  train_scaf=set(scaffold[train])
  return sum(1 for t in test if scaffold[t] in train_scaf)
print("random split test indices:", test_r)
print("their scaffold labels:", scaffold[test_r])
print("test molecule with training analogue:",count(test_r,train_r))

test_s = np.where((scaffold==2) | (scaffold==3))[0]
train_s = np.where((scaffold!=0) | (scaffold!=1))[0]
print("scaffold split test indices:", test_s)
print("their scaffold labels:", scaffold[test_s])
print("test molecule with training analogue:",count(test_s,train_s))

random split test indices: [ 2  7 10 11]
their scaffold labels: [0 2 3 3]
test molecule with training analogue: 2
scaffold split test indices: [ 7  8  9 10 11]
their scaffold labels: [2 2 2 3 3]
test molecule with training analogue: 5


Random split mesures interpolation near known scaffolds. Scaffold split measures generalisation to unseen frameworks. Both errors are worth reporting because they answer different deployement scenarios


Question:5

In [37]:
from math import comb

cases=[ (2,2,8,2),
        (5,1,3,2)]

def update(alpha,beta,s,f):
  a_post=alpha+s
  b_post=beta+f
  return a_post, b_post, a_post/(a_post+b_post)

for alpha,beta,s,f in cases:
  a_post,b_post,mu_post=update(alpha,beta,s,f)
  prior_mean=alpha/(alpha+beta)
  mle=(s/(s+f))
  print(f"prior parameters({alpha},{beta}) prior_mean={prior_mean:.4f}")
  print(f"posterior parameters({a_post},{b_post}) "
        f"posterior mean ={mu_post:.4f} mle={mle:.4f}")

p=0.8
prob = comb(5,4) * p**4 * (1-p)**1
print("P(4 or 5 sites active):", prob)

prior parameters(2,2) prior_mean=0.5000
posterior parameters(10,4) posterior mean =0.7143 mle=0.8000
prior parameters(5,1) prior_mean=0.8333
posterior parameters(8,3) posterior mean =0.7273 mle=0.6000
P(4 or 5 sites active): 0.4096


The post mean lies between the prior mean and the mle. It is a weighted compromise , which is pulled or moved towards whichever carries more effective count